# 01_translate_svg

Second step in the Revelation SVG translation workflow.

This notebook loads the translation units created by `00_open_and_inspect_svg.ipynb`, sends those units to an LLM in packetized JSON requests, validates the returned translations, saves translated JSON output, and writes translated SVG files.

This notebook assumes notebook 00 has already handled the Revelation-specific intake work: confirming the Illustrator SVG export settings, extracting SVG text units, applying preprocessing filters, and saving `json_files/translation_units.json`.

The overall pattern follows the earlier BST (*Bible Structure and Timeline*) and TBE SVG workflows, but this Revelation version includes bespoke translation prompts, robust SVG path resolution, and optional fragmented-`<tspan>` reconstruction handling for these graphics.

Major steps:

- configure source and target languages, model, API client, and optional domain context
- load `json_files/translation_units.json`
- packetize translation units for LLM requests
- send and validate one test packet
- run all packets and save timestamped translated JSON
- apply translations back into SVG files
- export CSV, XLSX, and Markdown review tables
- optionally write collapsed-tspan SVG outputs to `svg_output_files_collapsed_tspans/`

In [1]:
# Set languages and LLM model

source_language = 'English' # for LLM prompting
target_language = 'Spanish' # for LLM prompting

gemini_model_name = 'gemini-3.1-pro-preview' # Primary translator


In [2]:
# Load API key
# API key must be in a .env file in working directory

import os
from dotenv import load_dotenv
from google import genai

# Load .env once
load_dotenv()

# Force Gemini key usage (paid access)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found in environment")

# OPTIONAL: prevent accidental fallback
# This avoids the SDK silently choosing GOOGLE_API_KEY
os.environ.pop("GOOGLE_API_KEY", None)

# Create a single reusable client
genai_client = genai.Client(api_key=GEMINI_API_KEY)

# print(f"Gemini client initialized with key: {GEMINI_API_KEY[:4]}…{GEMINI_API_KEY[-4:]}")


## System message
**CAUTION:** Do not customize the following cell

In [3]:
# this is a bespoke customization specific for the Revelation graphic

def build_svg_units_system_instruction(
    source_language: str,
    target_language: str,
    domain_context: str | None = None,
) -> str:
    custom_block = ""
    if domain_context:
        custom_block = f"""
   - Domain context:
     {domain_context}
""".rstrip()

    return f"""
You are a professional translator with expertise in Bible, theology, and Christian educational materials.

You will receive ONE JSON object with this structure:

{{
  "units": [
    {{
      "unit_key": "<string>",
      "group_stack": "<string describing the SVG layer/group context>",
      "source_text": "<text in {source_language}>"
    }},
    ...
  ]
}}

Translate each `source_text` from {source_language} into {target_language}.
The `group_stack` is context to help you choose appropriate wording, capitalization, and abbreviations.

Most units are short SVG labels, single words, brief phrases, Bible-related headings, theological/expository labels, or short excerpts connected to the book of Revelation.

You must respond with ONE valid JSON object of the form:

{{
  "units": [
    {{
      "unit_key": "<same unit_key as input>",
      "translated_text": "<translated text in {target_language}>"
    }},
    ...
  ]
}}

REQUIREMENTS

1) JSON contract:
   - Return exactly one top-level JSON object with exactly one key: "units".
   - "units" must have the same number of entries as the input, in the same order.
   - Each output entry must contain exactly two keys: "unit_key" and "translated_text".
   - Do not include "group_stack" or "source_text" in the output.
   - Output must be valid JSON: no trailing commas, no comments, no code fences, no extra text.
   - Each "unit_key" must appear exactly once in the output. Do not duplicate, omit, or invent unit_key values.
   - The i-th output unit must correspond to the i-th input unit (same order).

2) Text handling:
   - Translate ordinary words and phrases naturally into {target_language}.
   - Prefer wording appropriate for Bible study, theology, and Christian educational graphics.
   - Preserve numbers, punctuation, verse markers, and abbreviations when meaningful.
   - Keep proper nouns unchanged unless a standard {target_language} form exists.
   - If a standard {target_language} form exists for a biblical book, biblical place, or biblical name, use the standard form.
   - If `source_text` is already primarily in {target_language}, copy it unchanged.
   - Use `group_stack` only as context; do not translate or reproduce the group_stack itself.{custom_block}

3) Bible references, numbers, and mixed text:
   - Preserve all numbers exactly as they appear in the source_text.
   - Preserve chapter:verse markers such as "2:7", "3:20", and "11:15-19" exactly.
   - When source_text begins with a verse marker followed by text, preserve the marker and translate the text.
   - When source_text contains both words and numbers, translate only the words and keep all numbers unchanged and in the same relative position.
   - Do not convert numbers to words, do not reformat them, and do not add or remove numeric content.
   - Preserve Bible reference notation unless the book name is ordinary text that should be translated in context.
   - Preserve compact structural or outline labels when they function as codes, such as letters, primes, ordinals, and numeric markers.

4) Short graphic labels:
   - These are short labels for an SVG graphic. Prefer concise translations that fit similar space.
   - Do not expand short labels into explanatory sentences.
   - Preserve capitalization style when appropriate, but use natural capitalization for {target_language}.
   - Translate repeated terms consistently across units.
   - For repeated labels such as "cross-reference" or "cross-references", translate the phrase consistently and preserve any leading number unchanged.

Your entire response must be ONLY the JSON object described above.
""".strip()

### Domain context (optional)
- Update `domain_context` below if this SVG set needs brief project-specific translation guidance.
- Keep it short and focused on terminology or interpretive context specific to this project.

In [4]:
# this is a bespoke customization specific for the Revelation graphic

primary_system_message = build_svg_units_system_instruction(
    source_language=source_language,
    target_language=target_language,
    domain_context=(
        "This material is an SVG graphic about the book of Revelation, including "
        "Bible-related labels, theological/expository headings, church names, "
        "cross-reference labels, section titles, and brief verse excerpts. "
        "Use terminology natural to a Protestant and Evangelical Christian context. "
        "Use established Biblical names, book names, and standard theological terms "
        "in the target language when they exist. "
        "Most units are short graphic labels, so keep translations concise and avoid "
        "expanding them into explanatory sentences. "
        "Preserve numbers, chapter:verse markers, outline letters, primes, ordinals, "
        "and structural codes when they function as notation. "
        "When a unit begins with a chapter:verse marker followed by text, preserve "
        "the marker exactly and translate the text. "
        "For repeated labels such as cross-reference or cross-references, translate "
        "the phrase consistently and preserve any leading number unchanged."
    ),
)

In [5]:
print(primary_system_message)

You are a professional translator with expertise in Bible, theology, and Christian educational materials.

You will receive ONE JSON object with this structure:

{
  "units": [
    {
      "unit_key": "<string>",
      "group_stack": "<string describing the SVG layer/group context>",
      "source_text": "<text in English>"
    },
    ...
  ]
}

Translate each `source_text` from English into Spanish.
The `group_stack` is context to help you choose appropriate wording, capitalization, and abbreviations.

Most units are short SVG labels, single words, brief phrases, Bible-related headings, theological/expository labels, or short excerpts connected to the book of Revelation.

You must respond with ONE valid JSON object of the form:

{
  "units": [
    {
      "unit_key": "<same unit_key as input>",
      "translated_text": "<translated text in Spanish>"
    },
    ...
  ]
}

REQUIREMENTS

1) JSON contract:
   - Return exactly one top-level JSON object with exactly one key: "units".
   - "

In [6]:
print(repr(primary_system_message))

'You are a professional translator with expertise in Bible, theology, and Christian educational materials.\n\nYou will receive ONE JSON object with this structure:\n\n{\n  "units": [\n    {\n      "unit_key": "<string>",\n      "group_stack": "<string describing the SVG layer/group context>",\n      "source_text": "<text in English>"\n    },\n    ...\n  ]\n}\n\nTranslate each `source_text` from English into Spanish.\nThe `group_stack` is context to help you choose appropriate wording, capitalization, and abbreviations.\n\nMost units are short SVG labels, single words, brief phrases, Bible-related headings, theological/expository labels, or short excerpts connected to the book of Revelation.\n\nYou must respond with ONE valid JSON object of the form:\n\n{\n  "units": [\n    {\n      "unit_key": "<same unit_key as input>",\n      "translated_text": "<translated text in Spanish>"\n    },\n    ...\n  ]\n}\n\nREQUIREMENTS\n\n1) JSON contract:\n   - Return exactly one top-level JSON object w

In [7]:
print("Domain context present:", "Protestant and Evangelical Christian context" in primary_system_message)

Domain context present: True


In [8]:
print("Using system message with length:", len(primary_system_message))

Using system message with length: 4438


In [9]:
def gemini_generate(payload_json_str: str, timeout: int = 120):
    return genai_client.models.generate_content(
        model=gemini_model_name,
        contents=payload_json_str,
        system_instruction=primary_system_message,
        request_options={"timeout": timeout},
    )

print("Gemini client ready:", genai_client)


Gemini client ready: <google.genai.client.Client object at 0x00000245ED16EF50>


## Import JSON payload

In [10]:
# Set directories

from pathlib import Path

PROJECT_ROOT = Path.cwd()
SVG_SOURCE_DIR = PROJECT_ROOT / "svg_source_files"
SVG_OUTPUT_DIR = PROJECT_ROOT / "svg_output_files"
JSON_DIR = PROJECT_ROOT / "json_files"

assert SVG_SOURCE_DIR.exists(), f"Missing folder: {SVG_SOURCE_DIR}"
assert SVG_OUTPUT_DIR.exists(), f"Missing folder: {SVG_OUTPUT_DIR}"
assert JSON_DIR.exists(), f"Missing folder: {JSON_DIR}"

print("PROJECT_ROOT:", PROJECT_ROOT.name)
print("SVG_SOURCE_DIR:", SVG_SOURCE_DIR.relative_to(PROJECT_ROOT.parent))
print("JSON_DIR:", JSON_DIR.relative_to(PROJECT_ROOT.parent))
print("SVG_OUTPUT_DIR:", SVG_OUTPUT_DIR.relative_to(PROJECT_ROOT.parent))

PROJECT_ROOT: rev
SVG_SOURCE_DIR: rev\svg_source_files
JSON_DIR: rev\json_files
SVG_OUTPUT_DIR: rev\svg_output_files


In [11]:
import json
from pathlib import Path
import pandas as pd

# --- locate and load the translation-units JSON produced in the previous notebook ---

units_path = JSON_DIR / "translation_units.json"
assert units_path.exists(), f"Missing file: {units_path}"

units = json.loads(units_path.read_text(encoding="utf-8"))
assert isinstance(units, list), "Expected a JSON array of unit records."

df_units = pd.DataFrame(units)
required_cols = {"unit_key", "unit_type", "source_file", "group_stack", "element_path", "source_text"}
missing = required_cols - set(df_units.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

df_units["source_text"] = df_units["source_text"].fillna("").astype(str)
df_units = df_units[df_units["source_text"].str.strip().ne("")].copy()
df_units.reset_index(drop=True, inplace=True)

print("Loaded units:", len(df_units))
print("Files:", df_units["source_file"].nunique())
df_units.head(10)


Loaded units: 164
Files: 1


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,translation_action,translation_note,target_text,contains_bible_reference,bible_reference_category,bible_reference_note,repeated_phrase_note
0,4041ecaccb42dee3d7f29c75e00b137f86bf10cc,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[14],None,None,None,232 cross-references,translate,,None,False,None,,"Repeated phrase detected: ""cross-reference(s)""..."
1,6bb999b7f7c222c530baaa3de7036f1e146d9037,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[15],None,None,None,All life insea dies,translate,,None,False,None,,
2,2e46ca8ff13d19e17a83d5683f948b0b18842b2d,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[17],None,None,None,Water toblood,translate,,None,False,None,,
3,fc6ae5198a7f6b72c6c870f6aae0f115be396976,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[19],None,None,None,Darkness,translate,,None,False,None,,
4,7fb015fbd2ea5ae5e874ec080f2ca02f6d409f96,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[1],None,None,None,7 Bowls,translate,,None,False,None,,
5,767d4c888fd38ad40e3da18fbd44fe1028462a9d,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[2],None,None,None,Revelation 16:1-16,translate,,None,False,None,,"Repeated term detected: ""Revelation"". Translat..."
6,bfb9c7cd67160a69813c8e27f1e43fd4bbf4aa0d,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[3],None,None,None,Sores,translate,,None,False,None,,
7,df69ae52bbb2968210fd3a6d360efe7d9ee9d95b,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[4],None,None,None,Fire from Sun,translate,,None,False,None,,
8,ddda055f5e30270491f9f708008bc3d314eb4ead,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[5],None,None,None,Euphratesdries,translate,,None,False,None,,
9,7efd525a141cb688c74f7d019c566b81654562ee,text,StructureOfRevelation.svg,_x37__churches,svg/g#_x37__churches/text[1],None,None,None,7 Churches,translate,,None,False,None,,


In [12]:
df_units["translation_action"].value_counts(dropna=False)

translation_action
translate    164
Name: count, dtype: int64

In [13]:
display(
    df_units[df_units["translation_action"].ne("translate")]
)

,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,translation_action,translation_note,target_text,contains_bible_reference,bible_reference_category,bible_reference_note,repeated_phrase_note


In [14]:
display(
    df_units[
        [
            "source_text",
            "translation_action",
            "bible_reference_category",
            "bible_reference_note",
            "repeated_phrase_note",
        ]
    ].sample(20, random_state=42)
)

,source_text,translation_action,bible_reference_category,bible_reference_note,repeated_phrase_note
135,Revelation 4The Throne,translate,None,,"Repeated term detected: ""Revelation"". Translat..."
115,D,translate,None,,
131,Future Glory,translate,None,,
55,(10x10x10),translate,None,,
95,Seven Victories,translate,None,,
29,Apocalypse,translate,None,,
157,Revelation 10 – 11:14,translate,None,,"Repeated term detected: ""Revelation"". Translat..."
51,7 Condemnations,translate,None,,
101,5,translate,None,,
145,Final Judgment,translate,None,,


## Prepare units
### Packet sizing
- These settings control how translation units are grouped into API requests.
- In most cases, leave these values unchanged.  
Only reduce them if requests are failing, timing out, or returning malformed/incomplete JSON.  
Increase them only if you are intentionally testing larger batches and understand the risk of oversized requests or harder-to-debug failures.
- A small final packet is normal and expected with this packing method.

In [15]:
# --- packetize units into request-sized chunks (simple char-based packing) ---
# Adjust these if you want larger/smaller packets; char-based is predictable across models.

# MAX_CHARS_PER_PACKET = 12000   # conservative for prompts + JSON overhead
# MAX_UNITS_PER_PACKET = 200     # hard cap to keep responses manageable

MAX_CHARS_PER_PACKET = 6000
MAX_UNITS_PER_PACKET = 100

def packetize_units(df: pd.DataFrame,
                    max_chars: int = MAX_CHARS_PER_PACKET,
                    max_units: int = MAX_UNITS_PER_PACKET):
    packets = []
    current = []
    current_chars = 0

    # Stable ordering helps reproducibility and diffing
    df_sorted = df.sort_values(["source_file", "group_stack", "element_path", "unit_type", "tspan_idx"], na_position="last")

    for _, r in df_sorted.iterrows():
        rec = {
            "unit_key": r["unit_key"],
            "group_stack": r["group_stack"],
            "source_text": r["source_text"],
        }
        # Estimate size as JSON string length
        rec_chars = len(json.dumps(rec, ensure_ascii=False))
        if rec_chars > max_chars:
            raise ValueError(f"Single unit exceeds max_chars ({rec_chars} > {max_chars}): {rec['unit_key']}")

        # Start new packet if needed
        if current and ((current_chars + rec_chars) > max_chars or (len(current) >= max_units)):
            packets.append(current)
            current = []
            current_chars = 0

        current.append(rec)
        current_chars += rec_chars

    if current:
        packets.append(current)

    return packets

packets = packetize_units(df_units)

print("Packets:", len(packets))
print("Units per packet (first 10):", [len(p) for p in packets[:10]])
print("Approx chars per packet (first 3):", [sum(len(json.dumps(u, ensure_ascii=False)) for u in p) for p in packets[:3]])


Packets: 4
Units per packet (first 10): [48, 49, 48, 19]
Approx chars per packet (first 3): [5937, 5982, 5893]


In [16]:
# --- build request payloads ready to send to Gemini (each payload is a single JSON object) ---
request_payloads = [{"units": p} for p in packets]

# quick peek at one payload
print("Example payload keys:", request_payloads[0].keys())
print("Example payload unit count:", len(request_payloads[0]["units"]))
print(json.dumps(request_payloads[0]["units"][0], ensure_ascii=False, indent=2))


Example payload keys: dict_keys(['units'])
Example payload unit count: 48
{
  "unit_key": "4041ecaccb42dee3d7f29c75e00b137f86bf10cc",
  "group_stack": "_x37__bowls",
  "source_text": "232 cross-references"
}


In [17]:
# inspect an entry

request_payloads[2]["units"][1]

{'unit_key': '77f2fa3682f7b95a2af719ff88e1b28160dfc818',
 'group_stack': 'chiasm',
 'source_text': '7:'}

### Inspect payload
- Pick a packet with a small number of entries using `packet_index`

In [18]:
# inspect payload
# pick one with a small number of entries

packet_index = 1 #4

request_payload = {"units": packets[packet_index]}

request_debug = {
    "model": gemini_model_name,
    "system_instruction": primary_system_message,
    "contents": request_payload,   # dict, not string
}

from pprint import pprint
pprint(request_debug)

{'contents': {'units': [{'group_stack': '_x37__victories_and_condemnations',
                         'source_text': '70 cross-references',
                         'unit_key': '43413f937123a0772a7ccce8a36d4b3a29689f5a'},
                        {'group_stack': '_x37__victories_and_condemnations',
                         'source_text': '126 cross-references',
                         'unit_key': '3347808ddb222697ca92634ad6310ee0ac4dee63'},
                        {'group_stack': '_x37__victories_and_condemnations',
                         'source_text': '152 cross-references',
                         'unit_key': 'c7dabab179dccfa4cef394de8d7af761a0c37bf4'},
                        {'group_stack': '_x37__victories_and_condemnations',
                         'source_text': '7 Condemnations',
                         'unit_key': 'cd3b612e280cf7fb2ae0e9e5ff54142edf5fa589'},
                        {'group_stack': '_x37__victories_and_condemnations',
                         'source_text

### Send one test packet

In [19]:
import json
import time
from google.genai import types

print("Preparing request...")
print("Model:", gemini_model_name)
print("Units in payload:", len(request_payload["units"]))
print("Sending request to Gemini...")

t0 = time.time()

response = genai_client.models.generate_content(
    model=gemini_model_name,
    contents=json.dumps(request_payload, ensure_ascii=False),
    config=types.GenerateContentConfig(
        system_instruction=primary_system_message,
    ),
)

elapsed = time.time() - t0
print(f"Response received in {elapsed:.2f} seconds")

print("Preview of response text:")
print(response.text[:2000])

Preparing request...
Model: gemini-3.1-pro-preview
Units in payload: 49
Sending request to Gemini...
Response received in 33.47 seconds
Preview of response text:
```json
{
  "units": [
    {
      "unit_key": "43413f937123a0772a7ccce8a36d4b3a29689f5a",
      "translated_text": "70 referencias cruzadas"
    },
    {
      "unit_key": "3347808ddb222697ca92634ad6310ee0ac4dee63",
      "translated_text": "126 referencias cruzadas"
    },
    {
      "unit_key": "c7dabab179dccfa4cef394de8d7af761a0c37bf4",
      "translated_text": "152 referencias cruzadas"
    },
    {
      "unit_key": "cd3b612e280cf7fb2ae0e9e5ff54142edf5fa589",
      "translated_text": "7 condenaciones"
    },
    {
      "unit_key": "3210776819b6c1e7853a9610994227204ff3e15a",
      "translated_text": "105 referencias cruzadas"
    },
    {
      "unit_key": "ad7dfdf45a25a85963f2fd080cd88274aa776f7c",
      "translated_text": "49 referencias cruzadas"
    },
    {
      "unit_key": "fc1754e88b05314fe27032b50f0795709f5aeba

In [20]:
# strip code fences
# One-off cleanup + parse for Gemini responses wrapped in ```json ... ```
import json

raw = response.text

# Strip leading/trailing whitespace first
s = raw.strip()

# If wrapped in fenced code block, remove the fences
if s.startswith("```"):
    lines = s.splitlines()
    # drop first line: ``` or ```json
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    # drop last line: ```
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    s = "\n".join(lines).strip()

# Now parse JSON
parsed = json.loads(s)

# Basic validation
assert "units" in parsed, "Missing top-level key 'units'"
assert len(parsed["units"]) == len(request_payload["units"]), (
    f"Unit count mismatch: got {len(parsed['units'])}, expected {len(request_payload['units'])}"
)

print("Parsed OK | units:", len(parsed["units"]))
print("First 2:", parsed["units"][:2])

Parsed OK | units: 49
First 2: [{'unit_key': '43413f937123a0772a7ccce8a36d4b3a29689f5a', 'translated_text': '70 referencias cruzadas'}, {'unit_key': '3347808ddb222697ca92634ad6310ee0ac4dee63', 'translated_text': '126 referencias cruzadas'}]


In [21]:
# verify packets (input vs model output)
def validate_units(result: dict, input_payload: dict):
    out_units = result.get("units", [])
    in_units = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    assert len(out_keys) == len(in_keys), f"Count mismatch: out={len(out_keys)} in={len(in_keys)}"
    assert out_keys == in_keys, "unit_key sequence mismatch (duplicates, missing, or re-ordered keys)."

# use the variables you actually have right now
validate_units(parsed, request_payload)
print("Validation OK")


Validation OK


### Run all packets
#### Load helper functions

In [22]:
from collections import Counter

def debug_unit_key_mismatch(result: dict, input_payload: dict, n_show: int = 20):
    out_units = result.get("units", [])
    in_units  = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    out_counts = Counter(out_keys)
    in_counts  = Counter(in_keys)

    missing = [k for k in in_counts if out_counts[k] == 0]
    extra   = [k for k in out_counts if in_counts[k] == 0]
    dup_out = [k for k, c in out_counts.items() if c > 1]
    dup_in  = [k for k, c in in_counts.items() if c > 1]

    print("Input units:", len(in_keys), "| Output units:", len(out_keys))
    print("Missing keys (in input but not output):", len(missing))
    print("Extra keys (in output but not input):", len(extra))
    print("Duplicates in output:", len(dup_out))
    print("Duplicates in input:", len(dup_in))

    if missing:
        print("\nFirst missing keys:")
        for k in missing[:n_show]:
            print("  ", k)

    if extra:
        print("\nFirst extra keys:")
        for k in extra[:n_show]:
            print("  ", k)

    if dup_out:
        print("\nFirst duplicated output keys:")
        for k in dup_out[:n_show]:
            print("  ", k, "count=", out_counts[k])

    # show first index where order differs (if lengths match)
    if len(out_keys) == len(in_keys):
        for i, (ok, ik) in enumerate(zip(out_keys, in_keys)):
            if ok != ik:
                print(f"\nFirst order mismatch at index {i}:")
                print("  expected:", ik)
                print("  got     :", ok)
                break


In [23]:
from collections import Counter

def debug_unit_key_mismatch(result: dict, input_payload: dict, n_show: int = 20):
    out_units = result.get("units", [])
    in_units  = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    out_counts = Counter(out_keys)
    in_counts  = Counter(in_keys)

    missing = [k for k in in_counts if out_counts[k] == 0]
    extra   = [k for k in out_counts if in_counts[k] == 0]
    dup_out = [k for k, c in out_counts.items() if c > 1]
    dup_in  = [k for k, c in in_counts.items() if c > 1]

    print("Input units:", len(in_keys), "| Output units:", len(out_keys))
    print("Missing keys (in input but not output):", len(missing))
    print("Extra keys (in output but not input):", len(extra))
    print("Duplicates in output:", len(dup_out))
    print("Duplicates in input:", len(dup_in))

    if missing:
        print("\nFirst missing keys:")
        for k in missing[:n_show]:
            print("  ", k)

    if extra:
        print("\nFirst extra keys:")
        for k in extra[:n_show]:
            print("  ", k)

    if dup_out:
        print("\nFirst duplicated output keys:")
        for k in dup_out[:n_show]:
            print("  ", k, "count=", out_counts[k])

    # show first index where order differs (if lengths match)
    if len(out_keys) == len(in_keys):
        for i, (ok, ik) in enumerate(zip(out_keys, in_keys)):
            if ok != ik:
                print(f"\nFirst order mismatch at index {i}:")
                print("  expected:", ik)
                print("  got     :", ok)
                break


In [24]:
import json
import time
import re
import pandas as pd
import httpx
from google.genai import types, errors as genai_errors

def extract_json_object(text: str) -> dict:
    t = (text or "").strip()
    t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s*```$", "", t)
    return json.loads(t)

def generate_with_retry(client, model_name: str, system_prompt: str, payload_obj: dict,
                        retries: int = 4, backoff_s: float = 5.0):
    payload_str = json.dumps(payload_obj, ensure_ascii=False)
    last_err = None

    for attempt in range(1, retries + 1):
        try:
            return client.models.generate_content(
                model=model_name,
                contents=payload_str,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                ),
            )

        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            last_err = e
            status = getattr(e, "status_code", None)
            msg = str(e)
            transient = (
                status in {429, 500, 503, 504}
                or "RESOURCE_EXHAUSTED" in msg
                or "UNAVAILABLE" in msg
                or "DEADLINE" in msg
            )
            if (attempt == retries) or (not transient):
                raise
            print(f"retry {attempt}/{retries} after API error: {type(e).__name__}: {e}")
            time.sleep(backoff_s * attempt)

        except (httpx.RemoteProtocolError, httpx.ReadTimeout, httpx.ConnectTimeout,
                httpx.ConnectError, httpx.NetworkError, httpx.TransportError) as e:
            last_err = e
            if attempt == retries:
                raise
            print(f"retry {attempt}/{retries} after transport error: {type(e).__name__}: {e}")
            time.sleep(backoff_s * attempt)

    raise last_err

In [25]:
from collections import Counter

def validate_units_keys_only(result: dict, input_payload: dict):
    """
    Validates that output contains exactly the same unit_keys as input,
    with the same counts, regardless of ordering.
    """
    out_units = result.get("units", [])
    in_units  = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    assert None not in out_keys, "Output contains a unit missing unit_key."
    assert None not in in_keys,  "Input contains a unit missing unit_key."

    out_counts = Counter(out_keys)
    in_counts  = Counter(in_keys)

    missing = [k for k in in_counts if out_counts[k] == 0]
    extra   = [k for k in out_counts if in_counts[k] == 0]
    wrong_count = [k for k in in_counts if out_counts[k] != in_counts[k]]

    assert not missing, f"Missing unit_keys in output (first 5): {missing[:5]}"
    assert not extra,   f"Extra unit_keys in output (first 5): {extra[:5]}"
    assert not wrong_count, f"Count mismatch for some keys (first 5): {wrong_count[:5]}"

def reorder_units_to_input_order(result: dict, input_payload: dict) -> dict:
    """
    Reorders result["units"] to match the order of input_payload["units"].
    Assumes validate_units_keys_only has passed.
    """
    out_units = result.get("units", [])
    in_units  = input_payload["units"]

    buckets = {}
    for u in out_units:
        k = u["unit_key"]
        buckets.setdefault(k, []).append(u)

    ordered = []
    for u_in in in_units:
        k = u_in["unit_key"]
        ordered.append(buckets[k].pop(0))

    leftovers = sum(len(v) for v in buckets.values())
    assert leftovers == 0, "Internal reorder error: leftover output units exist."

    return {"units": ordered}

In [26]:
# --- run all packets ---
all_out = []

for i, payload in enumerate(request_payloads):
    print(f"Packet {i+1}/{len(request_payloads)} ...", end=" ", flush=True)

    try:
        resp = generate_with_retry(
            client=genai_client,
            model_name=gemini_model_name,
            system_prompt=primary_system_message,
            payload_obj=payload,
            retries=4,
            backoff_s=5,
        )

        result = extract_json_object(resp.text)

        validate_units_keys_only(result, payload)
        result = reorder_units_to_input_order(result, payload)

        all_out.extend(result["units"])

        print("OK")

    except Exception as e:
        print(f"FAILED | {type(e).__name__}: {e}")
        raise

df_trans = pd.DataFrame(all_out)
print("Total translated units:", len(df_trans))
df_trans.head(10)

Packet 1/4 ... OK
Packet 2/4 ... OK
Packet 3/4 ... OK
Packet 4/4 ... OK
Total translated units: 164


,unit_key,translated_text
0,4041ecaccb42dee3d7f29c75e00b137f86bf10cc,232 referencias cruzadas
1,6bb999b7f7c222c530baaa3de7036f1e146d9037,Toda vida en el mar muere
2,2e46ca8ff13d19e17a83d5683f948b0b18842b2d,Agua en sangre
3,fc6ae5198a7f6b72c6c870f6aae0f115be396976,Tinieblas
4,7fb015fbd2ea5ae5e874ec080f2ca02f6d409f96,7 Copas
5,767d4c888fd38ad40e3da18fbd44fe1028462a9d,Apocalipsis 16:1-16
6,bfb9c7cd67160a69813c8e27f1e43fd4bbf4aa0d,Úlceras
7,df69ae52bbb2968210fd3a6d360efe7d9ee9d95b,Fuego del sol
8,ddda055f5e30270491f9f708008bc3d314eb4ead,Éufrates se seca
9,7efd525a141cb688c74f7d019c566b81654562ee,7 Iglesias


### Save translated phrases to timestamped json file

In [27]:
from datetime import datetime
import re
import json

def slugify_for_filename(s: str) -> str:
    """
    Make a filesystem-friendly slug from an arbitrary label.
    - Converts any non-alphanumeric/underscore runs to a single underscore
    - Trims leading/trailing underscores
    """
    s = (s or "").strip()
    return re.sub(r"[^\w]+", "_", s).strip("_")

def timestamp_yyyymmdd_hhmm(dt: datetime | None = None) -> str:
    dt = dt or datetime.now()
    return dt.strftime("%Y%m%d_%H%M")

ts = timestamp_yyyymmdd_hhmm()
lang_slug = slugify_for_filename(target_language)

translations_out_path = JSON_DIR / f"translations_{lang_slug}_{ts}.json"

print("lang_slug:", lang_slug)
print("timestamp:", ts)

translations_out_path.write_text(
    json.dumps(all_out, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Wrote translations:", translations_out_path.relative_to(PROJECT_ROOT))

lang_slug: Spanish
timestamp: 20260520_2017
Wrote translations: json_files\translations_Spanish_20260520_2017.json


## Swap out phrases and save to svg

In [28]:
# unit_key → translated_text (FROM MEMORY)
trans_map = {r["unit_key"]: r["translated_text"] for r in all_out}

print("Translations in memory:", len(trans_map))


Translations in memory: 164


In [29]:
import pandas as pd

trans_map = {r["unit_key"]: r["translated_text"] for r in all_out}

df_apply = df_units.copy()
df_apply["translated_text"] = df_apply["unit_key"].map(trans_map)

print("Rows in df_apply:", len(df_apply))
print("Translated rows:", df_apply["translated_text"].notna().sum())
print("Missing translations:", df_apply["translated_text"].isna().sum())

df_apply[["source_file", "unit_key", "unit_type", "element_path", "tspan_idx", "translated_text"]].head(10)

Rows in df_apply: 164
Translated rows: 164
Missing translations: 0


,source_file,unit_key,unit_type,element_path,tspan_idx,translated_text
0,StructureOfRevelation.svg,4041ecaccb42dee3d7f29c75e00b137f86bf10cc,text,svg/g#_x37__bowls/text[14],None,232 referencias cruzadas
1,StructureOfRevelation.svg,6bb999b7f7c222c530baaa3de7036f1e146d9037,text,svg/g#_x37__bowls/text[15],None,Toda vida en el mar muere
2,StructureOfRevelation.svg,2e46ca8ff13d19e17a83d5683f948b0b18842b2d,text,svg/g#_x37__bowls/text[17],None,Agua en sangre
3,StructureOfRevelation.svg,fc6ae5198a7f6b72c6c870f6aae0f115be396976,text,svg/g#_x37__bowls/text[19],None,Tinieblas
4,StructureOfRevelation.svg,7fb015fbd2ea5ae5e874ec080f2ca02f6d409f96,text,svg/g#_x37__bowls/text[1],None,7 Copas
5,StructureOfRevelation.svg,767d4c888fd38ad40e3da18fbd44fe1028462a9d,text,svg/g#_x37__bowls/text[2],None,Apocalipsis 16:1-16
6,StructureOfRevelation.svg,bfb9c7cd67160a69813c8e27f1e43fd4bbf4aa0d,text,svg/g#_x37__bowls/text[3],None,Úlceras
7,StructureOfRevelation.svg,df69ae52bbb2968210fd3a6d360efe7d9ee9d95b,text,svg/g#_x37__bowls/text[4],None,Fuego del sol
8,StructureOfRevelation.svg,ddda055f5e30270491f9f708008bc3d314eb4ead,text,svg/g#_x37__bowls/text[5],None,Éufrates se seca
9,StructureOfRevelation.svg,7efd525a141cb688c74f7d019c566b81654562ee,text,svg/g#_x37__churches/text[1],None,7 Iglesias


In [30]:
from datetime import datetime
from pathlib import Path
from lxml import etree
import re

SVG_NS = "http://www.w3.org/2000/svg"
NS = {"svg": SVG_NS}

target_language_slug = re.sub(r"[^A-Za-z0-9]+", "_", target_language).strip("_").lower()
timestamp = datetime.now().strftime("%Y%m%d_%H%M")


def localname(tag) -> str:
    if not isinstance(tag, str):
        return ""
    return tag.split("}", 1)[1] if tag.startswith("{") else tag

def find_by_element_path(root, path: str):
    """
    Resolve paths of the form:
    svg/g[1]/text[3]
    svg/g#Layer_1/text#text42
    svg/g[2]/text[1]/tspan[3]
    """
    if not path:
        return None

    parts = path.split("/")
    cur = root

    # first part should describe the root svg
    first = parts[0]
    if not first.startswith("svg"):
        return None

    for part in parts[1:]:
        m = re.fullmatch(r"([A-Za-z0-9_:-]+)(?:#(.+)|\[(\d+)\])?", part)
        if not m:
            return None

        tag, el_id, idx = m.groups()
        candidates = [
            child for child in cur
            if isinstance(getattr(child, "tag", None), str) and localname(child.tag) == tag
        ]

        if el_id is not None:
            match = None
            for child in candidates:
                if child.get("id") == el_id:
                    match = child
                    break
            if match is None:
                return None
            cur = match

        elif idx is not None:
            idx0 = int(idx) - 1  # stored paths are 1-based
            if idx0 < 0 or idx0 >= len(candidates):
                return None
            cur = candidates[idx0]

        else:
            if not candidates:
                return None
            cur = candidates[0]

    return cur


def find_by_element_path_with_id_fallback(root, path: str):
    """Resolve a stored SVG element_path, falling back to descendant ID lookups."""
    el = find_by_element_path(root, path)
    if el is not None:
        return el
    if not path:
        return None

    parts = path.split("/")
    if not parts or not parts[0].startswith("svg"):
        return None

    parsed_parts = []
    for part in parts[1:]:
        m = re.fullmatch(r"([A-Za-z0-9_:-]+)(?:#(.+)|\[(\d+)\])?", part)
        if not m:
            return None
        parsed_parts.append(m.groups())

    id_positions = [i for i, (_, el_id, _) in enumerate(parsed_parts) if el_id is not None]
    if not id_positions:
        return None

    cur = root
    for id_position in id_positions:
        tag, el_id, _ = parsed_parts[id_position]
        match = None
        for candidate in cur.iter():
            if not isinstance(getattr(candidate, "tag", None), str):
                continue
            if localname(candidate.tag) == tag and candidate.get("id") == el_id:
                match = candidate
                break
        if match is None:
            return None
        cur = match

    for tag, el_id, idx in parsed_parts[id_positions[-1] + 1:]:
        candidates = [
            child for child in cur
            if isinstance(getattr(child, "tag", None), str) and localname(child.tag) == tag
        ]

        if el_id is not None:
            cur = next((child for child in candidates if child.get("id") == el_id), None)
            if cur is None:
                return None
        elif idx is not None:
            idx0 = int(idx) - 1  # stored paths are 1-based
            if idx0 < 0 or idx0 >= len(candidates):
                return None
            cur = candidates[idx0]
        else:
            if not candidates:
                return None
            cur = candidates[0]

    return cur


def set_text_preserve_none(el, new_text: str):
    """
    Replace text content safely.
    """
    el.text = "" if new_text is None else str(new_text)


def split_svg_prolog_open_tag_and_tail(svg_text: str):
    match = re.search(r"<svg\b[^>]*>", svg_text, flags=re.DOTALL)
    if not match:
        raise ValueError("Could not find opening <svg> tag.")
    return svg_text[:match.start()], svg_text[match.start():match.end()], svg_text[match.end():]


def write_svg_preserving_original_header(tree, source_svg_path: Path, out_path: Path) -> None:
    """Write SVG using the original Illustrator prolog and opening <svg> tag."""
    original_text = source_svg_path.read_text(encoding="utf-8", errors="replace")
    original_header, original_svg_open_tag, _ = split_svg_prolog_open_tag_and_tail(original_text)
    root = tree.getroot()
    root_text = root.text or ""
    serialized_children = "".join(
        etree.tostring(child, encoding="unicode", pretty_print=False)
        for child in root
    )
    serialized_svg_inner = root_text + serialized_children
    serialized_svg = original_header + original_svg_open_tag + serialized_svg_inner + f"</{localname(root.tag)}>"

    print("original header characters:", len(original_header))
    print("original opening svg tag characters:", len(original_svg_open_tag))
    print("output path:", out_path)

    out_path.write_text(serialized_svg, encoding="utf-8")


def apply_translations_to_svg(
    svg_path: Path,
    units_for_file: pd.DataFrame,
    target_language_slug: str,
    timestamp: str,
    output_dir: Path,
) -> Path:
    parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
    tree = etree.parse(str(svg_path), parser)
    root = tree.getroot()

    updated = 0
    not_found = 0
    skipped_blank = 0

    # sort for reproducibility and to process parent text before tspans consistently
    units_for_file = units_for_file.sort_values(
        ["element_path", "unit_type", "tspan_idx"],
        na_position="last"
    )

    for _, u in units_for_file.iterrows():
        new_text = u["translated_text"]
        if pd.isna(new_text):
            skipped_blank += 1
            continue

        el = find_by_element_path_with_id_fallback(root, u["element_path"])
        if el is None:
            not_found += 1
            continue

        if u["unit_type"] == "text":
            set_text_preserve_none(el, new_text)
            updated += 1

        elif u["unit_type"] == "tspan":
            idx = u.get("tspan_idx")
            if pd.isna(idx):
                not_found += 1
                continue

            tspans = el.findall(".//svg:tspan", namespaces=NS)
            idx = int(idx)

            if idx < 0 or idx >= len(tspans):
                not_found += 1
                continue

            set_text_preserve_none(tspans[idx], new_text)
            updated += 1

    out_path = output_dir / f"{svg_path.stem}_{target_language_slug}_{timestamp}{svg_path.suffix}"
    write_svg_preserving_original_header(tree, svg_path, out_path)

    print(
        f"{svg_path.name} -> {out_path.name} | "
        f"updated={updated} | not_found={not_found} | skipped_blank={skipped_blank}"
    )
    return out_path

In [31]:
diagnostic_file = df_apply["source_file"].dropna().iloc[0]
diagnostic_units = df_apply[df_apply["source_file"].eq(diagnostic_file)].head(10).copy()
diagnostic_svg_path = SVG_SOURCE_DIR / diagnostic_file

parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
tree = etree.parse(str(diagnostic_svg_path), parser)
root = tree.getroot()

print("SVG path:", diagnostic_svg_path)
print("Rows tested:", len(diagnostic_units))

found_count = 0
for _, row in diagnostic_units.iterrows():
    el = find_by_element_path_with_id_fallback(root, row["element_path"])
    found = el is not None
    found_count += int(found)
    print("source_text:", row["source_text"])
    print("element_path:", row["element_path"])
    print("found:", found)

print(f"Found count: {found_count} out of {len(diagnostic_units)}")

for wanted_id in ["Artboard_1", "Cards", "Ephesus_card"]:
    matches = root.xpath(f'.//*[@id="{wanted_id}"]')
    print(wanted_id, len(matches), [localname(m.tag) for m in matches[:3]])


SVG path: c:\Users\drhan\OneDrive\Documents\bb_github\resources\workflows\translation\svg\rev\svg_source_files\StructureOfRevelation.svg
Rows tested: 10
source_text: 232 cross-references
element_path: svg/g#_x37__bowls/text[14]
found: True
source_text: All life insea dies
element_path: svg/g#_x37__bowls/text[15]
found: True
source_text: Water toblood
element_path: svg/g#_x37__bowls/text[17]
found: True
source_text: Darkness
element_path: svg/g#_x37__bowls/text[19]
found: True
source_text: 7 Bowls
element_path: svg/g#_x37__bowls/text[1]
found: True
source_text: Revelation 16:1-16
element_path: svg/g#_x37__bowls/text[2]
found: True
source_text: Sores
element_path: svg/g#_x37__bowls/text[3]
found: True
source_text: Fire from Sun
element_path: svg/g#_x37__bowls/text[4]
found: True
source_text: Euphratesdries
element_path: svg/g#_x37__bowls/text[5]
found: True
source_text: 7 Churches
element_path: svg/g#_x37__churches/text[1]
found: True
Found count: 10 out of 10
Artboard_1 0 []
Cards 0 []


In [32]:
df_apply["unit_type"].value_counts(dropna=False)

unit_type
text    164
Name: count, dtype: int64

In [33]:
df_apply[df_apply["translated_text"].notna()][
    ["source_file", "unit_type", "element_path", "tspan_idx", "source_text", "translated_text"]
].head(20)

,source_file,unit_type,element_path,tspan_idx,source_text,translated_text
0,StructureOfRevelation.svg,text,svg/g#_x37__bowls/text[14],None,232 cross-references,232 referencias cruzadas
1,StructureOfRevelation.svg,text,svg/g#_x37__bowls/text[15],None,All life insea dies,Toda vida en el mar muere
2,StructureOfRevelation.svg,text,svg/g#_x37__bowls/text[17],None,Water toblood,Agua en sangre
3,StructureOfRevelation.svg,text,svg/g#_x37__bowls/text[19],None,Darkness,Tinieblas
4,StructureOfRevelation.svg,text,svg/g#_x37__bowls/text[1],None,7 Bowls,7 Copas
5,StructureOfRevelation.svg,text,svg/g#_x37__bowls/text[2],None,Revelation 16:1-16,Apocalipsis 16:1-16
6,StructureOfRevelation.svg,text,svg/g#_x37__bowls/text[3],None,Sores,Úlceras
7,StructureOfRevelation.svg,text,svg/g#_x37__bowls/text[4],None,Fire from Sun,Fuego del sol
8,StructureOfRevelation.svg,text,svg/g#_x37__bowls/text[5],None,Euphratesdries,Éufrates se seca
9,StructureOfRevelation.svg,text,svg/g#_x37__churches/text[1],None,7 Churches,7 Iglesias


In [34]:
out_files = []

for source_file, units_for_file in df_apply.groupby("source_file", sort=True):
    svg_path = SVG_SOURCE_DIR / source_file
    out_file = apply_translations_to_svg(
        svg_path=svg_path,
        units_for_file=units_for_file,
        target_language_slug=target_language_slug,
        timestamp=timestamp,
        output_dir=SVG_OUTPUT_DIR,
    )
    out_files.append(out_file)

print("Wrote files:", len(out_files))
for p in out_files:
    print(" -", p.relative_to(PROJECT_ROOT))

original header characters: 134
original opening svg tag characters: 196
output path: c:\Users\drhan\OneDrive\Documents\bb_github\resources\workflows\translation\svg\rev\svg_output_files\StructureOfRevelation_spanish_20260520_2017.svg
StructureOfRevelation.svg -> StructureOfRevelation_spanish_20260520_2017.svg | updated=164 | not_found=0 | skipped_blank=0
Wrote files: 1
 - svg_output_files\StructureOfRevelation_spanish_20260520_2017.svg


In [35]:
# debug

test_file = df_apply["source_file"].iloc[0]
test_units = df_apply[df_apply["source_file"].eq(test_file)].copy()

svg_path = SVG_SOURCE_DIR / test_file
parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
tree = etree.parse(str(svg_path), parser)
root = tree.getroot()

print("SVG path:", svg_path)
print("Exists:", svg_path.exists())
print("Root tag:", root.tag)
print("First source_file:", test_file)
print("Rows for file:", len(test_units))

sample = test_units[["source_text", "unit_type", "element_path", "translated_text"]].head(10)
display(sample)

for _, row in sample.iterrows():
    el = find_by_element_path_with_id_fallback(root, row["element_path"])
    print("\nSOURCE:", row["source_text"])
    print("PATH:", row["element_path"])
    print("FOUND:", el is not None, "| TAG:", localname(el.tag) if el is not None else None)

SVG path: c:\Users\drhan\OneDrive\Documents\bb_github\resources\workflows\translation\svg\rev\svg_source_files\StructureOfRevelation.svg
Exists: True
Root tag: {http://www.w3.org/2000/svg}svg
First source_file: StructureOfRevelation.svg
Rows for file: 164


,source_text,unit_type,element_path,translated_text
0,232 cross-references,text,svg/g#_x37__bowls/text[14],232 referencias cruzadas
1,All life insea dies,text,svg/g#_x37__bowls/text[15],Toda vida en el mar muere
2,Water toblood,text,svg/g#_x37__bowls/text[17],Agua en sangre
3,Darkness,text,svg/g#_x37__bowls/text[19],Tinieblas
4,7 Bowls,text,svg/g#_x37__bowls/text[1],7 Copas
5,Revelation 16:1-16,text,svg/g#_x37__bowls/text[2],Apocalipsis 16:1-16
6,Sores,text,svg/g#_x37__bowls/text[3],Úlceras
7,Fire from Sun,text,svg/g#_x37__bowls/text[4],Fuego del sol
8,Euphratesdries,text,svg/g#_x37__bowls/text[5],Éufrates se seca
9,7 Churches,text,svg/g#_x37__churches/text[1],7 Iglesias



SOURCE: 232 cross-references
PATH: svg/g#_x37__bowls/text[14]
FOUND: True | TAG: text

SOURCE: All life insea dies
PATH: svg/g#_x37__bowls/text[15]
FOUND: True | TAG: text

SOURCE: Water toblood
PATH: svg/g#_x37__bowls/text[17]
FOUND: True | TAG: text

SOURCE: Darkness
PATH: svg/g#_x37__bowls/text[19]
FOUND: True | TAG: text

SOURCE: 7 Bowls
PATH: svg/g#_x37__bowls/text[1]
FOUND: True | TAG: text

SOURCE: Revelation 16:1-16
PATH: svg/g#_x37__bowls/text[2]
FOUND: True | TAG: text

SOURCE: Sores
PATH: svg/g#_x37__bowls/text[3]
FOUND: True | TAG: text

SOURCE: Fire from Sun
PATH: svg/g#_x37__bowls/text[4]
FOUND: True | TAG: text

SOURCE: Euphratesdries
PATH: svg/g#_x37__bowls/text[5]
FOUND: True | TAG: text

SOURCE: 7 Churches
PATH: svg/g#_x37__churches/text[1]
FOUND: True | TAG: text


In [36]:
# debug

print("Immediate g children under root:")
for i, child in enumerate(
    [c for c in root if isinstance(getattr(c, "tag", None), str) and localname(c.tag) == "g"],
    start=1
):
    print(i, child.get("id"))

Immediate g children under root:
1 margins_1_x2F_4_x22_
2 guides
3 white_background
4 interlude_background
5 border
6 grid
7 _x37__churches
8 throne_room
9 _x37__seals
10 _x37__trumpets
11 _x37__visions
12 _x37__bowls
13 _x37__victories_and_condemnations
14 interlude
15 future_glory
16 chiasm
17 logo_cc
18 endurance_asterisks


In [37]:
# debug

print("Root children:")
for i, child in enumerate(list(root)[:30], start=1):
    if isinstance(getattr(child, "tag", None), str):
        print(i, localname(child.tag), child.get("id"))

Root children:
1 font None
2 font None
3 font None
4 font None
5 font None
6 font None
7 font None
8 font None
9 font None
10 font None
11 font None
12 pattern _x36__lpi_10_x25_
13 g margins_1_x2F_4_x22_
14 g guides
15 g white_background
16 g interlude_background
17 g border
18 g grid
19 g _x37__churches
20 g throne_room
21 g _x37__seals
22 g _x37__trumpets
23 g _x37__visions
24 g _x37__bowls
25 g _x37__victories_and_condemnations
26 g interlude
27 g future_glory
28 g chiasm
29 g logo_cc
30 g endurance_asterisks


### Export translation table

In [38]:
from datetime import datetime
import pandas as pd
import re

def slugify_for_filename(s: str) -> str:
    s = (s or "").strip()
    return re.sub(r"[^\w]+", "_", s).strip("_").lower()

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
lang_slug = slugify_for_filename(target_language)

# Build export dataframe
df_export = (
    df_apply.loc[:, ["source_text", "translated_text", "group_stack"]]
    .copy()
)

print("Rows in export df:", len(df_export))
print("Missing translated_text:", df_export["translated_text"].isna().sum())

# Output paths
csv_path = SVG_OUTPUT_DIR / f"translations_table_{lang_slug}_{timestamp}.csv"
xlsx_path = SVG_OUTPUT_DIR / f"translations_table_{lang_slug}_{timestamp}.xlsx"
md_path = SVG_OUTPUT_DIR / f"translations_table_{lang_slug}_{timestamp}.md"

# Write files
df_export.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_export.to_excel(xlsx_path, index=False)

md_text = df_export.to_markdown(index=False)
md_path.write_text(md_text, encoding="utf-8")

print("Wrote CSV :", csv_path.relative_to(PROJECT_ROOT))
print("Wrote XLSX:", xlsx_path.relative_to(PROJECT_ROOT))
print("Wrote MD  :", md_path.relative_to(PROJECT_ROOT))

df_export.head(10)

Rows in export df: 164
Missing translated_text: 0
Wrote CSV : svg_output_files\translations_table_spanish_20260520_2017.csv
Wrote XLSX: svg_output_files\translations_table_spanish_20260520_2017.xlsx
Wrote MD  : svg_output_files\translations_table_spanish_20260520_2017.md


,source_text,translated_text,group_stack
0,232 cross-references,232 referencias cruzadas,_x37__bowls
1,All life insea dies,Toda vida en el mar muere,_x37__bowls
2,Water toblood,Agua en sangre,_x37__bowls
3,Darkness,Tinieblas,_x37__bowls
4,7 Bowls,7 Copas,_x37__bowls
5,Revelation 16:1-16,Apocalipsis 16:1-16,_x37__bowls
6,Sores,Úlceras,_x37__bowls
7,Fire from Sun,Fuego del sol,_x37__bowls
8,Euphratesdries,Éufrates se seca,_x37__bowls
9,7 Churches,7 Iglesias,_x37__churches


## Collapse fragmented tspans

This section is used for text-level translation units whose original SVG text was fragmented across multiple `<tspan>` elements. It reuses the notebook's existing robust path resolver, `find_by_element_path_with_id_fallback`, and only defines collapse-specific reconstruction helpers.

This does not replace the general SVG apply function. It provides an alternate output pass that writes each translated text-level unit into its matched `<text>` element, preserving all existing `<tspan>` elements by default for Illustrator compatibility. The first `<tspan>` receives the translated text, and extra original tspans are blanked unless removal is explicitly enabled.


In [39]:
# Collapsed-tspan outputs use the same output directory and filename tokens as the
# general SVG apply step, with an added suffix to distinguish this reconstruction pass.
collapsed_target_language_slug = target_language_slug
collapsed_timestamp = timestamp
collapsed_output_dir = SVG_OUTPUT_DIR
remove_extra_tspans = False

print("Collapsed-tspan output dir:", collapsed_output_dir.relative_to(PROJECT_ROOT))


Collapsed-tspan output dir: svg_output_files


In [40]:
def collapsed_direct_tspans(text_el):
    """Return direct child tspans for a matched <text> element."""
    return [
        child for child in text_el
        if isinstance(getattr(child, "tag", None), str) and localname(child.tag) == "tspan"
    ]


def collapsed_replace_text_content(text_el, translated_text: str, remove_extra_tspans: bool = False) -> dict:
    """Replace visible text while preserving Illustrator tspan structure by default."""
    direct_tspans = collapsed_direct_tspans(text_el)
    blanked_extra_tspans = 0
    removed_extra_tspans = 0
    collapsed_fragmented_text = False

    if direct_tspans:
        first_tspan = direct_tspans[0]
        first_tspan.text = translated_text
        first_tspan.tail = None
        text_el.text = None

        for extra_tspan in direct_tspans[1:]:
            if remove_extra_tspans:
                text_el.remove(extra_tspan)
                removed_extra_tspans += 1
            else:
                extra_tspan.text = ""
                if extra_tspan.tail is not None:
                    extra_tspan.tail = ""
                blanked_extra_tspans += 1

        collapsed_fragmented_text = len(direct_tspans) > 1
    else:
        set_text_preserve_none(text_el, translated_text)

    return {
        "collapsed_fragmented_text": collapsed_fragmented_text,
        "blanked_extra_tspans": blanked_extra_tspans,
        "removed_extra_tspans": removed_extra_tspans,
    }


def collapsed_text_rows(units_for_file: pd.DataFrame) -> pd.DataFrame:
    """Return text-level rows that have nonblank translated_text values."""
    work = units_for_file.copy()
    if "unit_type" in work.columns:
        work = work[work["unit_type"].fillna("").astype(str).eq("text")].copy()

    return work[work["element_path"].notna()].copy()


In [41]:
def collapsed_apply_text_level_translations_to_svg(
    svg_path: Path,
    units_for_file: pd.DataFrame,
    target_language_slug: str,
    timestamp: str,
    output_dir: Path,
    remove_extra_tspans: bool = False,
) -> tuple[Path, dict]:
    """
    Collapse-aware reconstruction for text-level SVG units.

    This reuses find_by_element_path_with_id_fallback for path resolution and only
    changes how matched <text> elements with fragmented tspans are rewritten.
    """
    parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
    tree = etree.parse(str(svg_path), parser)
    root = tree.getroot()

    text_rows = collapsed_text_rows(units_for_file).sort_values("element_path")

    updated = 0
    fragmented_collapsed = 0
    blanked_extra_tspans = 0
    removed_extra_tspans = 0
    unresolved_paths = 0
    skipped_blank = 0

    for _, row in text_rows.iterrows():
        translated_text = row["translated_text"]
        if pd.isna(translated_text) or str(translated_text).strip() == "":
            skipped_blank += 1
            continue

        text_el = find_by_element_path_with_id_fallback(root, row["element_path"])
        if text_el is None or localname(text_el.tag) != "text":
            unresolved_paths += 1
            continue

        collapse_stats = collapsed_replace_text_content(
            text_el,
            str(translated_text),
            remove_extra_tspans=remove_extra_tspans,
        )
        updated += 1
        fragmented_collapsed += int(collapse_stats["collapsed_fragmented_text"])
        blanked_extra_tspans += collapse_stats["blanked_extra_tspans"]
        removed_extra_tspans += collapse_stats["removed_extra_tspans"]

    out_path = output_dir / f"{svg_path.stem}_{target_language_slug}_{timestamp}_collapsed_tspans{svg_path.suffix}"
    write_svg_preserving_original_header(tree, svg_path, out_path)

    stats = {
        "source_svg_filename": svg_path.name,
        "output_svg_filename": out_path.name,
        "text_elements_updated": updated,
        "fragmented_text_elements_collapsed": fragmented_collapsed,
        "extra_tspans_blanked": blanked_extra_tspans,
        "extra_tspans_removed": removed_extra_tspans,
        "unresolved_paths": unresolved_paths,
        "skipped_blank_translated_text": skipped_blank,
    }
    return out_path, stats


def collapsed_print_reconstruction_stats(stats: dict) -> None:
    """Print notebook-friendly diagnostics for one collapsed-tspan reconstruction run."""
    print("source SVG filename:", stats["source_svg_filename"])
    print("output SVG filename:", stats["output_svg_filename"])
    print("number of text elements updated:", stats["text_elements_updated"])
    print("number of fragmented text elements collapsed:", stats["fragmented_text_elements_collapsed"])
    print("number of extra tspans blanked:", stats["extra_tspans_blanked"])
    print("number of extra tspans removed:", stats["extra_tspans_removed"])
    print("number of unresolved paths:", stats["unresolved_paths"])
    print("number skipped because translated_text was blank:", stats["skipped_blank_translated_text"])


In [42]:
# Manual diagnostic: inspect likely fragmented rows before running the collapse apply pass.
diagnostic_candidates = collapsed_text_rows(df_apply)
diagnostic_candidates = diagnostic_candidates[
    diagnostic_candidates["translated_text"].notna()
    & diagnostic_candidates["translated_text"].astype(str).str.strip().ne("")
].copy()

diagnostic_source_file = next(
    source_file for source_file in sorted(diagnostic_candidates["source_file"].dropna().unique())
    if (SVG_SOURCE_DIR / source_file).exists()
)
diagnostic_svg_path = SVG_SOURCE_DIR / diagnostic_source_file
parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
tree = etree.parse(str(diagnostic_svg_path), parser)
root = tree.getroot()

diagnostic_rows = diagnostic_candidates[
    diagnostic_candidates["source_file"].eq(diagnostic_source_file)
].head(10)

for _, row in diagnostic_rows.iterrows():
    el = find_by_element_path_with_id_fallback(root, row["element_path"])
    tspan_count = len(collapsed_direct_tspans(el)) if el is not None and localname(el.tag) == "text" else 0
    print("source_text:", row["source_text"])
    print("translated_text:", row["translated_text"])
    print("element_path:", row["element_path"])
    print("resolver_found:", el is not None)
    print("tspans_before_collapse:", tspan_count)
    print()


source_text: 232 cross-references
translated_text: 232 referencias cruzadas
element_path: svg/g#_x37__bowls/text[14]
resolver_found: True
tspans_before_collapse: 0

source_text: All life insea dies
translated_text: Toda vida en el mar muere
element_path: svg/g#_x37__bowls/text[15]
resolver_found: True
tspans_before_collapse: 2

source_text: Water toblood
translated_text: Agua en sangre
element_path: svg/g#_x37__bowls/text[17]
resolver_found: True
tspans_before_collapse: 2

source_text: Darkness
translated_text: Tinieblas
element_path: svg/g#_x37__bowls/text[19]
resolver_found: True
tspans_before_collapse: 0

source_text: 7 Bowls
translated_text: 7 Copas
element_path: svg/g#_x37__bowls/text[1]
resolver_found: True
tspans_before_collapse: 0

source_text: Revelation 16:1-16
translated_text: Apocalipsis 16:1-16
element_path: svg/g#_x37__bowls/text[2]
resolver_found: True
tspans_before_collapse: 0

source_text: Sores
translated_text: Úlceras
element_path: svg/g#_x37__bowls/text[3]
resolver_

Run the optional full pass below after the manual diagnostic shows that the robust resolver is finding the expected `<text>` elements and the tspan counts match the fragmented rows you want to collapse.


In [43]:
# Optional full run: reconstruct every source SVG with the collapsed-tspan strategy.
# Run this cell after the manual diagnostic looks correct.
collapsed_out_files = []
collapsed_all_stats = []

for source_file, units_for_file in df_apply.groupby("source_file", sort=True):
    svg_path = SVG_SOURCE_DIR / source_file
    if not svg_path.exists():
        print("Skipping missing source SVG:", source_file)
        continue

    out_path, stats = collapsed_apply_text_level_translations_to_svg(
        svg_path=svg_path,
        units_for_file=units_for_file,
        target_language_slug=collapsed_target_language_slug,
        timestamp=collapsed_timestamp,
        output_dir=collapsed_output_dir,
        remove_extra_tspans=remove_extra_tspans,
    )
    collapsed_out_files.append(out_path)
    collapsed_all_stats.append(stats)
    collapsed_print_reconstruction_stats(stats)
    print()

print("Wrote collapsed-tspan files:", len(collapsed_out_files))
print("Total text elements updated:", sum(s["text_elements_updated"] for s in collapsed_all_stats))
print("Total fragmented text elements collapsed:", sum(s["fragmented_text_elements_collapsed"] for s in collapsed_all_stats))
print("Total extra tspans blanked:", sum(s["extra_tspans_blanked"] for s in collapsed_all_stats))
print("Total extra tspans removed:", sum(s["extra_tspans_removed"] for s in collapsed_all_stats))
print("Total unresolved paths:", sum(s["unresolved_paths"] for s in collapsed_all_stats))
print("Total skipped because translated_text was blank:", sum(s["skipped_blank_translated_text"] for s in collapsed_all_stats))


original header characters: 134
original opening svg tag characters: 196
output path: c:\Users\drhan\OneDrive\Documents\bb_github\resources\workflows\translation\svg\rev\svg_output_files\StructureOfRevelation_spanish_20260520_2017_collapsed_tspans.svg
source SVG filename: StructureOfRevelation.svg
output SVG filename: StructureOfRevelation_spanish_20260520_2017_collapsed_tspans.svg
number of text elements updated: 164
number of fragmented text elements collapsed: 35
number of extra tspans blanked: 52
number of extra tspans removed: 0
number of unresolved paths: 0
number skipped because translated_text was blank: 0

Wrote collapsed-tspan files: 1
Total text elements updated: 164
Total fragmented text elements collapsed: 35
Total extra tspans blanked: 52
Total extra tspans removed: 0
Total unresolved paths: 0
Total skipped because translated_text was blank: 0


In [44]:
# Check xml validity

from pathlib import Path
from lxml import etree

svg_file = Path("svg_output_files/StructureOfRevelation_spanish_20260520_1131_collapsed_tspans.svg") #"svg_output_files/SevenChurchesOfRevelation_spanish_20260520_1131_collapsed_tspans.svg"

try:
    parser = etree.XMLParser(recover=False, huge_tree=True)
    etree.parse(str(svg_file), parser)
    print("XML parse OK")
except etree.XMLSyntaxError as e:
    print("XML syntax error:")
    print(e)

XML parse OK


In [45]:
from pathlib import Path
import json
import pandas as pd

p = Path("json_files/translation_units.json")

with p.open("r", encoding="utf-8") as f:
    data = json.load(f)

if isinstance(data, dict) and "units" in data:
    units = data["units"]
elif isinstance(data, list):
    units = data
else:
    raise ValueError(f"Unexpected JSON structure: {type(data)}")

df_check = pd.DataFrame(units)

print("Rows:", len(df_check))
print("Columns:", list(df_check.columns))

if "translation_action" in df_check.columns:
    print(df_check["translation_action"].value_counts(dropna=False))

if "bible_reference_category" in df_check.columns:
    print(df_check["bible_reference_category"].value_counts(dropna=False))

display(df_check.head())

Rows: 164
Columns: ['unit_key', 'unit_type', 'source_file', 'group_stack', 'element_path', 'text_id', 'tspan_id', 'tspan_idx', 'source_text', 'translation_action', 'translation_note', 'target_text', 'contains_bible_reference', 'bible_reference_category', 'bible_reference_note', 'repeated_phrase_note']
translation_action
translate    164
Name: count, dtype: int64
bible_reference_category
None                   156
reference_with_text      8
Name: count, dtype: int64


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,translation_action,translation_note,target_text,contains_bible_reference,bible_reference_category,bible_reference_note,repeated_phrase_note
0,4041ecaccb42dee3d7f29c75e00b137f86bf10cc,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[14],None,None,None,232 cross-references,translate,,None,False,None,,"Repeated phrase detected: ""cross-reference(s)""..."
1,6bb999b7f7c222c530baaa3de7036f1e146d9037,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[15],None,None,None,All life insea dies,translate,,None,False,None,,
2,2e46ca8ff13d19e17a83d5683f948b0b18842b2d,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[17],None,None,None,Water toblood,translate,,None,False,None,,
3,fc6ae5198a7f6b72c6c870f6aae0f115be396976,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[19],None,None,None,Darkness,translate,,None,False,None,,
4,7fb015fbd2ea5ae5e874ec080f2ca02f6d409f96,text,StructureOfRevelation.svg,_x37__bowls,svg/g#_x37__bowls/text[1],None,None,None,7 Bowls,translate,,None,False,None,,
